# [Tool Call Accuracy Evaluator Sample](https://learn.microsoft.com/en-us/python/api/azure-ai-evaluation/azure.ai.evaluation.toolcallaccuracyevaluator?view=azure-python)

**IMPORTANT NOTE**<br/>
- These samples use `GPT-4.1 mini` because `azure-ai-evaluation 1.18.3` local agent evaluators send the legacy max_tokens parameter.
- Newer GPT-5 deployments require `max_completion_tokens` and aren't compatible with this local evaluator path.
- For managed evaluations with newer judge models, see 4 - cloud evaluation.

## Objective
This sample demonstrates to how to use tool call accuracy evaluator on agent data. The supported input formats include:
- simple data such as strings and `dict` describing tool calls;
- user-agent conversations in the form of list of agent messages.

## What this evaluator assesses
The Tool Call Accuracy evaluator assesses how accurately an AI uses tools by examining:
- Relevance to the conversation
- Parameter correctness according to tool definitions
- Parameter value extraction from the conversation
- Potential usefulness of the tool call

The evaluator uses a binary scoring (0 or 1) for each tool call:
    - Score 0: The tool call is irrelevant or contains information not in the conversation/definition
    - Score 1: The tool call is relevant with properly extracted parameters from the conversation

If there are multiple call, the final score will be an **average** of individual tool calls, which can be interpreted as the **passing rate** of tool calls.<br/>
This evaluation focuses on measuring whether tool calls meaningfully contribute to addressing query while properly following tool definitions and using information present in the conversation history.

## Required input
Tool Call Accuracy requires following input:
- Query - This can be a single query or a list of messages(conversation history with agent). Latter helps to determine if Agent used the information in history to make right tool calls.
- Tool Calls - Tool Call(s) made by Agent to answer the query. Optional - if response has tool calls, if not provided evaluator will look for tool calls in response.
- Response - (Optional) Response from Agent (or any GenAI App). This can be a single text response or a list or messages generated as part of Agent Response. If tool calls are not provide Tool Call Accuracy Evaluator will look at response for tool calls.
- Tool Definitions - Tool(s) definition used by Agent to answer the query. 


## Variables, Constants and Libraries definition

In [1]:
import os, sys
from dotenv import load_dotenv  # requires python-dotenv
from azure.identity import DefaultAzureCredential

if not load_dotenv("./../credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

openai_api_version = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
foundry_project_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")

# gpt-4.1-mini is the latest working model because later models require max_completion_tokens, while this evaluator sends max_tokens
azure_evaluation_compatible_deployment_name = os.environ[
    "AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME"
]

credential = DefaultAzureCredential()

print(f"azure_openai_endpoint: {azure_openai_endpoint}")
print(f"foundry_project_endpoint: {foundry_project_endpoint}")
print(f"azure_evaluation_compatible_deployment_name: {azure_evaluation_compatible_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

azure_openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
foundry_project_endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project
azure_evaluation_compatible_deployment_name: gpt-4.1-mini
openai_api_version: 2025-04-01-preview


### Initialize Tool Call Accuracy Evaluator


In [2]:
from azure.ai.evaluation import ToolCallAccuracyEvaluator, AzureOpenAIModelConfiguration
from pprint import pprint
import warnings

warnings.filterwarnings("ignore")

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_evaluation_compatible_deployment_name,
    api_version=openai_api_version,
)

tool_call_accuracy = ToolCallAccuracyEvaluator(
    model_config,
    credential=credential,
)

# Print some constants
print(f'openai endpoint: <{model_config["azure_endpoint"]}>')
print(f'azure deployment name: <{model_config["azure_deployment"]}>')
print(f'openai api version: <{model_config["api_version"]}>')

Class ToolCallAccuracyEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


openai endpoint: <https://mm-ai-upskilling-project-resourc.openai.azure.com/>
azure deployment name: <gpt-4.1-mini>
openai api version: <2025-04-01-preview>


### Samples

#### Evaluating Single Tool Call

In [3]:
query = "How is the weather in Seattle ?"
tool_call = {
    "type": "tool_call",
    "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
    "name": "fetch_weather",
    "arguments": {"location": "Seattle"},
}

tool_definition = {
    "id": "fetch_weather",
    "name": "fetch_weather",
    "description": "Fetches the weather information for the specified location.",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
    },
}

In [4]:
response = tool_call_accuracy(query=query, tool_calls=[tool_call], tool_definitions=[tool_definition])
pprint(response)

{'gpt_tool_call_accuracy': 5.0,
 'tool_call_accuracy': 5.0,
 'tool_call_accuracy_passed': True,
 'tool_call_accuracy_properties': {'completion_tokens': 305,
                                   'correct_tool_calls_made_by_agent': 1,
                                   'excess_tool_calls': {'details': [],
                                                         'total': 0},
                                   'finish_reason': 'stop',
                                   'missing_tool_calls': {'details': [],
                                                          'total': 0},
                                   'model': 'gpt-4.1-mini-2025-04-14',
                                   'per_tool_call_details': [{'correct_calls_made_by_agent': 1,
                                                              'correct_tool_percentage': 1.0,
                                                              'tool_call_errors': 0,
                                                              'tool_name': 'f

#### Multiple Tool Calls used by Agent to respond

In [5]:
query = "How is the weather in Seattle ?"
tool_calls = [
    {
        "type": "tool_call",
        "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
        "name": "fetch_weather",
        "arguments": {"location": "Seattle"},
    },
    {
        "type": "tool_call",
        "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
        "name": "fetch_weather",
        "arguments": {"location": "London"},
    },
]

tool_definition = {
    "id": "fetch_weather",
    "name": "fetch_weather",
    "description": "Fetches the weather information for the specified location.",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
    },
}

In [6]:
response = tool_call_accuracy(
    query=query, 
    tool_calls=tool_calls, 
    tool_definitions=[tool_definition])

pprint(response)

{'gpt_tool_call_accuracy': 3.0,
 'tool_call_accuracy': 3.0,
 'tool_call_accuracy_passed': True,
 'tool_call_accuracy_properties': {'completion_tokens': 376,
                                   'correct_tool_calls_made_by_agent': 1,
                                   'excess_tool_calls': {'details': [{'excess_count': 1,
                                                                      'tool_name': 'fetch_weather'}],
                                                         'total': 1},
                                   'finish_reason': 'stop',
                                   'missing_tool_calls': {'details': [],
                                                          'total': 0},
                                   'model': 'gpt-4.1-mini-2025-04-14',
                                   'per_tool_call_details': [{'correct_calls_made_by_agent': 1,
                                                              'correct_tool_percentage': 0.5,
                                           

#### Tool Calls passed as part of `Response` (common for agent case)
- Tool Call Accuracy Evaluator extracts tool calls from response

In [7]:
query = "Can you send me an email with weather information for Seattle?"
response = [
    {
        "createdAt": "2025-03-26T17:27:35Z",
        "run_id": "run_zblZyGCNyx6aOYTadmaqM4QN",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
                "name": "fetch_weather",
                "arguments": {"location": "Seattle"},
            }
        ],
    },
    {
        "createdAt": "2025-03-26T17:27:37Z",
        "run_id": "run_zblZyGCNyx6aOYTadmaqM4QN",
        "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
        "role": "tool",
        "content": [{"type": "tool_result", "tool_result": {"weather": "Rainy, 14\u00b0C"}}],
    },
    {
        "createdAt": "2025-03-26T17:27:38Z",
        "run_id": "run_zblZyGCNyx6aOYTadmaqM4QN",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "call_iq9RuPxqzykebvACgX8pqRW2",
                "name": "send_email",
                "arguments": {
                    "recipient": "your_email@example.com",
                    "subject": "Weather Information for Seattle",
                    "body": "The current weather in Seattle is rainy with a temperature of 14\u00b0C.",
                },
            }
        ],
    },
    {
        "createdAt": "2025-03-26T17:27:41Z",
        "run_id": "run_zblZyGCNyx6aOYTadmaqM4QN",
        "tool_call_id": "call_iq9RuPxqzykebvACgX8pqRW2",
        "role": "tool",
        "content": [
            {"type": "tool_result", "tool_result": {"message": "Email successfully sent to your_email@example.com."}}
        ],
    },
    {
        "createdAt": "2025-03-26T17:27:42Z",
        "run_id": "run_zblZyGCNyx6aOYTadmaqM4QN",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "I have successfully sent you an email with the weather information for Seattle. The current weather is rainy with a temperature of 14\u00b0C.",
            }
        ],
    },
]

tool_definitions = [
    {
        "name": "fetch_weather",
        "description": "Fetches the weather information for the specified location.",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
        },
    },
    {
        "name": "send_email",
        "description": "Sends an email with the specified subject and body to the recipient.",
        "parameters": {
            "type": "object",
            "properties": {
                "recipient": {"type": "string", "description": "Email address of the recipient."},
                "subject": {"type": "string", "description": "Subject of the email."},
                "body": {"type": "string", "description": "Body content of the email."},
            },
        },
    },
]

In [8]:
response = tool_call_accuracy(
    query=query, 
    response=response, 
    tool_definitions=tool_definitions)

pprint(response)

{'gpt_tool_call_accuracy': 5.0,
 'tool_call_accuracy': 5.0,
 'tool_call_accuracy_passed': True,
 'tool_call_accuracy_properties': {'completion_tokens': 393,
                                   'correct_tool_calls_made_by_agent': 2,
                                   'excess_tool_calls': {'details': [],
                                                         'total': 0},
                                   'finish_reason': 'stop',
                                   'missing_tool_calls': {'details': [],
                                                          'total': 0},
                                   'model': 'gpt-4.1-mini-2025-04-14',
                                   'per_tool_call_details': [{'correct_calls_made_by_agent': 1,
                                                              'correct_tool_percentage': 1.0,
                                                              'tool_call_errors': 0,
                                                              'tool_name': 'f

## Batch evaluation and optional Foundry publishing



`evaluate()` runs the evaluators over every record in the JSONL dataset and saves the aggregated results locally. The evaluator itself still calls the configured Azure OpenAI judge model.



Set `publish_to_foundry = True` in the next cell to also upload the completed evaluation results to Microsoft Foundry. Publishing requires a valid `FOUNDRY_PROJECT_ENDPOINT` in `credentials_my.env`; when publishing is disabled, no Foundry workspace is required.

In [9]:
from pathlib import Path
from azure.ai.evaluation import evaluate

# Keep this False for a local batch run. Set it to True to also publish the results to Foundry.
publish_to_foundry = True

project_endpoint = (
    foundry_project_endpoint.strip() or None
    if publish_to_foundry and foundry_project_endpoint
    else None
)

if publish_to_foundry and not project_endpoint:
    raise ValueError(
        "FOUNDRY_PROJECT_ENDPOINT must be set when publish_to_foundry is True."
    )

data_path = Path("evaluation_data.jsonl")
output_path = Path("evaluation_results/tool_call_accuracy.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

response = evaluate(
    data=data_path,
    evaluation_name="Tool Call Accuracy Evaluation",
    evaluators={
        "tool_call_accuracy": tool_call_accuracy,
    },
    azure_ai_project=project_endpoint,
    output_path=output_path,
)

# restore output_path to stdout
import sys
(type(sys.stdout).__module__, type(sys.stdout).__name__)

while hasattr(sys.stdout, "_prev_out"):
    sys.stdout = sys.stdout._prev_out

while hasattr(sys.stderr, "_prev_out"):
    sys.stderr = sys.stderr._prev_out

print(f"Local results: {output_path.resolve()}")
studio_url = response.get("studio_url")

if studio_url:
    print(f"Microsoft Foundry URL (to be checked on the classic Foundry portal): {studio_url}")

2026-08-11 16:49:14 +0200 272048956699008 execution.bulk     INFO     Finished 1 / 5 lines.
2026-08-11 16:49:14 +0200 272048956699008 execution.bulk     INFO     Average execution time for completed lines: 0.32 seconds. Estimated time for incomplete lines: 1.28 seconds.
2026-08-11 16:49:21 +0200 272048956699008 execution.bulk     INFO     Finished 2 / 5 lines.
2026-08-11 16:49:21 +0200 272048956699008 execution.bulk     INFO     Average execution time for completed lines: 3.65 seconds. Estimated time for incomplete lines: 10.95 seconds.
2026-08-11 16:49:22 +0200 272048956699008 execution.bulk     INFO     Finished 3 / 5 lines.
2026-08-11 16:49:22 +0200 272048956699008 execution.bulk     INFO     Average execution time for completed lines: 2.67 seconds. Estimated time for incomplete lines: 5.34 seconds.
2026-08-11 16:49:22 +0200 272048956699008 execution.bulk     INFO     Finished 4 / 5 lines.
2026-08-11 16:49:22 +0200 272048956699008 execution.bulk     INFO     Average execution time f

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "tool_call_accuracy_20260811_144914_288992"
Run status: "Completed"
Start time: "2026-08-11 14:49:14.288992+00:00"
Duration: "0:00:07.450928"

======= Combined Run Summary (Per Evaluator) =======

{
    "tool_call_accuracy": {
        "status": "Completed",
        "duration": "0:00:07.450928",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


Evaluation results saved to "/home/mauromi/git_repos/genai_evaluation/2 - local evaluation/evaluation_results/tool_call_accuracy.json".

Local results: /home/mauromi/git_repos/genai_evaluation/2 - local evaluation/evaluation_results/tool_call_accuracy.json
Microsoft Foundry URL (to be checked on the classic Foundry portal): https://ai.azure.com/resource/build/evaluation/e5c6957b-a1f0-4229-b1cd-18007f6dbfd9?wsid=/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/a